In [1]:

import mne
import mne_bids
import numpy as np
import pandas as pd
import pyprep
from nki_rs2_eeg import read_file , write_file
from nki_rs2_eeg.read_file import read_raw_nwb
from nki_rs2_eeg.config import (
    RAW_DATA_DIR,
    DERIVATIVES_DIR,
    SESSION_ID,
    TASK_ID,
    RUN_ID,
    CHANNEL_NAMES,
    FS,
    LINE_NOISE_HZ,
)

#mne.viz.set_3d_backend("pyvistaqt")


In [3]:
file = "/data3/cdb/MOBI_LAB/NKI_RS2/EEG_derivatives/sub-A00066860/ses-MOBI1A/eeg/sub-A00066860_ses-MOBI1A_task-passivepresent_run-1_eeg.edf"
raw = read_file.read_processed_edf(file)
raw, _ = write_file.trim_data_to_event(raw, "Onset Movie", "Offset Movie")
raw.set_channel_types({"Fp1": "eog", "Fp2": "eog"})
raw.set_eeg_reference("average")

Extracting EDF parameters from /data3/cdb/MOBI_LAB/NKI_RS2/EEG_derivatives/sub-A00066860/ses-MOBI1A/eeg/sub-A00066860_ses-MOBI1A_task-passivepresent_run-1_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 54249  =      0.000 ...   216.996 secs...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


<RawEDF | sub-A00066860_ses-MOBI1A_task-passivepresent_run-1_eeg.edf, 64 x 51755 (207.0 s), ~25.3 MiB, data loaded>

In [ ]:
raw.plot()

In [4]:
events, event_id = mne.events_from_annotations(raw)
eid = event_id['blink']
blink_epochs = mne.Epochs(raw=raw, events=events, event_id=eid, tmin=-0.5, tmax=0.5, preload=True)

Used Annotations descriptions: [np.str_('Offset Movie'), np.str_('Onset Movie'), np.str_('blink')]
Not setting metadata
93 matching events found
Setting baseline interval to [-0.5, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 93 events and 251 original time points ...
75 bad epochs dropped


In [5]:
model = mne.preprocessing.EOGRegression(picks="eeg", picks_artifact="eog").fit(blink_epochs)
raw_cleaned = model.apply(raw)

No projector specified for this dataset. Please consider the method self.add_proj.
No projector specified for this dataset. Please consider the method self.add_proj.


In [ ]:
raw_cleaned.plot()

In [ ]:
dirty, _ = read_file.read_raw_nwb("/Users/bryan.gonzalez/NKI_RS2_EEG/data/raw/sub-A00066860/ses-MOBI1A/sub-A00066860_ses-MOBI1A_task-passivepresent_run-01_MoBI.nwb")
dirty, _ = write_file.trim_data_to_event(dirty, "Onset Movie", "Offset Movie")

In [ ]:
dirty.plot()

In [ ]:
dirty.get_montage()

# Blink Regression

In [ ]:
# change the type to EOG for Fp1 and Fp2
raw.set_channel_types({"Fp1": "eog", "Fp2": "eog"})
raw.set_eeg_reference("average")
epochs = mne.make_fixed_length_epochs(raw, duration=0.5, preload=True)
epochs.set_montage(dirty.get_montage())
model_plain = mne.preprocessing.EOGRegression(picks="eeg", picks_artifact="eog").fit(epochs)
fig = model_plain.plot()

In [ ]:
raw_eog = raw.copy().pick_types(eeg=False, eog=True).get

raw_eeg = raw.copy().pick_types(eeg=True, eog=False)

b = np.linalg.inv(raw_eog @ raw_eog.T) @ raw_eog @ raw_eeg.T


In [ ]:
raw.set_eeg_reference("average")
epochs = mne.make_fixed_length_epochs(raw, duration=0.5, preload=True)
epochs.set_montage(dirty.get_montage())
model_plain = mne.preprocessing.EOGRegression(picks="eeg", picks_artifact=["Fp1", "Fp2"]).fit(epochs)
fig = model_plain.plot()


In [ ]:
eog_epochs = mne.preprocessing.create_eog_epochs(raw, ch_name=["Fp1", "Fp2"])
eog_epochs.set_montage(dirty.get_montage())
eog_evoked = eog_epochs.average("all")
eog_evoked.plot("all")

In [ ]:
# perform regression on the evoked blink response
model_evoked = mne.preprocessing.EOGRegression(picks="eeg", picks_artifact=["Fp1", "Fp2"]).fit(eog_evoked)
model_evoked.plot(vlim=(None, 0.4))

In [ ]:
# apply the regression coefficients to the original epochs
epochs_clean_evoked = model_evoked.apply(epochs)#.apply_baseline()
fig = epochs_clean_evoked.average("all").plot()

In [ ]:
epochs_clean_evoked.plot()

# Line Noise Removal

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_multitaper

def plot_parietal_psd_before_after_notch(raw, channel=None, fmin=1, fmax=120, bandwidth=2.0):
    """
    Plot power spectrum of a parietal channel before and after 60Hz notch filtering.
    Uses multitaper spectral decomposition.
    
    Parameters:
    -----------
    raw : mne.io.Raw
        MNE Raw object
    channel : str or None
        Specific channel name. If None, auto-selects first parietal channel.
    fmin : float
        Minimum frequency for PSD (default: 1 Hz)
    fmax : float
        Maximum frequency for PSD (default: 120 Hz)
    bandwidth : float
        Multitaper bandwidth in Hz (default: 2.0)
    
    Returns:
    --------
    fig : matplotlib.Figure
    channel_name : str
        The channel used
    """
    
    # ── 1. Select parietal channel ─────────────────────────────────────────────
    if channel is None:
        parietal_picks = mne.pick_channels_regexp(raw.ch_names, regexp=r'P[z0-9]|CP[z0-9]')
        if len(parietal_picks) == 0:
            raise ValueError("No parietal channels found. Specify a channel name manually.")
        channel = raw.ch_names[parietal_picks[0]]
        print(f"Auto-selected parietal channel: {channel}")
    
    ch_idx = raw.ch_names.index(channel)
    sfreq = raw.info['sfreq']
    
    # ── 2. Extract raw signal (before filtering) ───────────────────────────────
    data_before, _ = raw[ch_idx, :]          # shape: (1, n_times)
    data_before = data_before[0]             # flatten to (n_times,)
    print("Applying the multi-taper notch filter to the raw data to remove noise........")
    # ── 3. Apply notch filter at 60 Hz (and harmonics) ────────────────────────
    notch_freqs = np.arange(60, min(sfreq / 2, fmax + 1), 60)
    raw_clean = raw.copy().notch_filter(
        freqs=notch_freqs,
        method='spectrum_fit',      # <-- multitaper spectral decomposition
        mt_bandwidth=bandwidth,     # multitaper half-bandwidth (Hz)
        p_value=0.05,               # significance threshold for line detection
        picks=[ch_idx],
        verbose=False
    )
    data_after = raw_clean[ch_idx, :][0][0]
    print("Multi-taper notch filter applied successfully.")
    print("creating the power spectrum plot before and after notch filtering........")

    # ── 4. Multitaper PSD ──────────────────────────────────────────────────────
    psd_before, freqs = psd_array_multitaper(
        data_before[np.newaxis, :],
        sfreq=sfreq,
        fmin=fmin,
        fmax=fmax,
        bandwidth=bandwidth,
        normalization='full',
        verbose=False
    )
    psd_before = psd_before[0]   # (n_freqs,)

    psd_after, _ = psd_array_multitaper(
        data_after[np.newaxis, :],
        sfreq=sfreq,
        fmin=fmin,
        fmax=fmax,
        bandwidth=bandwidth,
        normalization='full',
        verbose=False
    )
    psd_after = psd_after[0]

    # Convert to dB
    psd_before_db = 10 * np.log10(psd_before)
    psd_after_db  = 10 * np.log10(psd_after)
    diff_db       = psd_after_db - psd_before_db
    print("Multitaper PSD computed for both before and after notch filtering.")

    print("Plotting the PSD before and after notch filtering with band shading and annotations........")
    # ── 5. Plot ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(11, 10), sharex=True)
    # fig.patch.set_facecolor('#0f1117')
    for ax in axes:
     #  ax.set_facecolor('#1a1d27')
        ax.tick_params(colors='#c9d1d9')
        ax.yaxis.label.set_color('#c9d1d9')
        ax.xaxis.label.set_color('#c9d1d9')
        ax.title.set_color('#e6edf3')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

    # EEG band shading
    bands = [
        ('Delta',  0.5,  4,  '#3b4a6b'),
        ('Theta',  4,    8,  '#3b5e4a'),
        ('Alpha',  8,    13, '#5e5a2e'),
        ('Beta',   13,   30, '#5e3b2e'),
        ('Gamma',  30,   80, '#4a2e5e'),
    ]
    for ax in axes[:1]:
        for bname, b1, b2, bcolor in bands:
            ax.axvspan(b1, b2, alpha=0.25, color=bcolor, zorder=0)
            mid = (b1 + b2) / 2
            ypos = ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -1
            ax.text(mid, ax.get_ylim()[0], bname, color='#8b949e',
                    fontsize=7, ha='center', va='bottom', style='italic')

    # Top panel: before & after PSD
    ax0 = axes[0]
    ax0.plot(freqs, psd_before_db, color='#58a6ff', lw=1.5,
             label='Before (original)', alpha=0.9)
    ax0.plot(freqs, psd_after_db,  color='#3fb950', lw=1.5,
             label='After multi-taper', alpha=0.9)

    # Mark 60 Hz harmonics
    notch_freqs = np.arange(60, fmax + 1, 60)
    for nf in notch_freqs:
        ax0.axvline(nf, color='#f85149', lw=0.8, ls='--', alpha=0.7)
        ax0.text(nf + 0.5, ax0.get_ylim()[1] if ax0.get_ylim()[1] != 1 else 0,
                 f'{int(nf)} Hz', color='#f85149', fontsize=7, va='top')

    # Band shading on top panel
    for bname, b1, b2, bcolor in bands:
        ax0.axvspan(b1, b2, alpha=0.18, color=bcolor, zorder=0)

    ax0.set_ylabel('Power (dB)', fontsize=11)
    ax0.set_title(f'Multitaper PSD — Channel {channel}  |  Bandwidth = {bandwidth} Hz',
                  fontsize=12, fontweight='bold', pad=10)
    ax0.legend(facecolor='#1a1d27', edgecolor='#30363d',
               labelcolor='#c9d1d9', fontsize=9, loc='upper right')
    ax0.grid(True, alpha=0.15, color='#8b949e')

    # Bottom panel: difference
    ax1 = axes[1]
    ax1.fill_between(freqs, diff_db, 0,
                     where=(diff_db < 0), color='#3fb950', alpha=0.6,
                     label='Power removed')
    ax1.fill_between(freqs, diff_db, 0,
                     where=(diff_db >= 0), color='#f0883e', alpha=0.6,
                     label='Power added (artifact)')
    ax1.axhline(0, color='#8b949e', lw=0.8, ls='-')
    for nf in notch_freqs:
        ax1.axvline(nf, color='#f85149', lw=0.8, ls='--', alpha=0.7)

    ax1.set_xlabel('Frequency (Hz)', fontsize=11)
    ax1.set_ylabel('Δ Power (dB)', fontsize=11)
    ax1.set_title('Difference (After − Before)', fontsize=11, pad=8)
    ax1.legend(facecolor='#1a1d27', edgecolor='#30363d',
               labelcolor='#c9d1d9', fontsize=9, loc='lower right')
    ax1.grid(True, alpha=0.15, color='#8b949e')

    plt.tight_layout(h_pad=1.5)
    plt.savefig('/home/bgonzalez/NKI_RS2_EEG/figures/psd_before_after_notch.png',
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f"Figure saved → /home/bgonzalez/NKI_RS2_EEG/figures/psd_before_after_notch.png")
    return fig, channel





In [ ]:
fig, ch = plot_parietal_psd_before_after_notch(raw_cropped, fmin=1, fmax=150, bandwidth=2.0)

In [ ]:
raw_cropped 

In [ ]:
raw_cropped.compute_psd(fmin=1, fmax=150, bandwidth=2.0, n_jobs=1, picks=['CP5']).plot()

In [ ]:
import seaborn as sns
def plot_raw_seaborn(
    raw: mne.io.BaseRaw,
    picks=None,          # channel names/indices/type string, e.g. 'eeg'
    tmin: float = 0.0,   # start time in seconds
    tmax: float = 5.0,   # end time in seconds
    scale: float = 1e6,  # unit scale (1e6 → µV for EEG)
    offset_scale: float = 1.0,  # vertical spacing multiplier between channels
):
    """
    Plot MNE Raw data with Seaborn.

    Parameters
    ----------
    raw          : mne.io.BaseRaw
    picks        : channel selector (None = all)
    tmin / tmax  : time window to plot (seconds)
    scale        : multiply signal (e.g. 1e6 to convert V → µV)
    offset_scale : how far apart channels are drawn (1 = 1 × peak amplitude)
    """
    # ── 1. Extract data ────────────────────────────────────────────────────
    picks_idx = mne.pick_types(raw.info, **{"eeg": True}) if picks is None else \
                mne.pick_channels(raw.ch_names, picks) if isinstance(picks, list) else \
                mne.pick_types(raw.info, **{picks: True}) if isinstance(picks, str) else picks

    data, times = raw[picks_idx, :]          # shape: (n_ch, n_times)
    ch_names = [raw.ch_names[i] for i in picks_idx]

    # Slice to requested time window
    mask = (times >= tmin) & (times <= tmax)
    data, times = data[:, mask] * scale, times[mask]

    # ── 2. Build tidy (long-form) DataFrame ────────────────────────────────
    n_ch = len(ch_names)
    peak = np.percentile(np.abs(data), 95, axis=1)          # robust amplitude
    offsets = np.arange(n_ch)[::-1] * peak.mean() * offset_scale * 2

    rows = []
    for i, (ch, off) in enumerate(zip(ch_names, offsets)):
        rows.append(
            pd.DataFrame({
                "time_s":   times,
                "amplitude": data[i] + off,
                "channel":  ch,
            })
        )
    df = pd.concat(rows, ignore_index=True)

    # ── 3. Seaborn plot ────────────────────────────────────────────────────
    sns.set_theme(style="darkgrid", palette="tab10")
    fig, ax = plt.subplots(figsize=(30, max(4, n_ch * 0.6)))

    sns.lineplot(
        data=df,
        x="time_s",
        y="amplitude",
        hue="channel",
        linewidth=0.8,
        legend=False,   # labels added manually as y-ticks for clarity
        ax=ax,
    )

    # Replace numeric y-ticks with channel names
    ax.set_yticks(offsets)
    ax.set_yticklabels(ch_names, fontsize=8)
    ax.set_xlabel("Time (s)", fontsize=11)
    ax.set_ylabel("Channel", fontsize=11)
    ax.set_title(
        f"Cleaned signal  |  {tmin:.1f} – {tmax:.1f} s  |  {n_ch} channels",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()
    return fig, ax



In [ ]:
fig, ax = plot_raw_seaborn(raw, picks="eeg", tmin=0, tmax=160, scale=1e6, offset_scale=1.0)

In [ ]:
def run_prep(raw: mne.io.Raw) -> pyprep.PrepPipeline:
    """Run the PREP pipeline for EEG preprocessing.
    
    Applies the PREP pipeline which includes:
    1. Setting channel montage
    2. Filtering (0-125 Hz) and resampling to 250 Hz
    3. Reference correction
    4. Line noise removal (60 Hz and harmonics)
    
    Args:
        raw (mne.io.Raw): The raw EEG data in MNE format.

    Returns:
        pyprep.PrepPipeline: The fitted PREP pipeline object containing
                             the cleaned data and preprocessing information.
    """
    raw.filter(l_freq=0, h_freq=125).resample(250)
    prep_params = {
        "ref_chs": "eeg",
        "reref_chs": "eeg",
        "line_freqs": np.arange(60, raw.info["sfreq"] / 2, 60),
    }
    prep = pyprep.PrepPipeline(
        raw, 
        montage=raw.get_montage(), 
        channel_wise=True, 
        prep_params=prep_params
    )

    return prep.fit()

In [ ]:
raw_cleaned = run_prep(raw)

# run pyPrep

In [ ]:
prep_output = run_prep(raw)
raw_cleaned = prep_output.raw

In [ ]:
fig, ax = plot_raw_seaborn(raw_cleaned, picks="eeg", tmin=0, tmax=160, scale=1e7, offset_scale=1.0)

In [ ]:
prep_output.still_noisy_channels
